In [1]:
!pip install folium spektral scikit-learn networkx

In [2]:
import pandas as pd
import numpy as np
import folium
import networkx as nx
from datetime import datetime

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from spektral.data import Graph
from spektral.layers import GCNConv
from spektral.data import Dataset, Loader
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, LSTM, Input, Concatenate, TimeDistributed, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam

In [3]:
df = pd.read_csv('/content/drive/MyDrive/GDG/Research Paper/Datasets/Clean_LA_Crime.csv')

In [4]:
df.head()

,DATE OCC,Day,Month,Year,Day of Week,Is_Weekend,Is_Holiday,TIME OCC,Hour,Minute,Time of Day,AREA,AREA NAME,Crm Cd,Crm Cd Desc,LOCATION,LAT,LON
0,2020-03-01,1,3,2020,Sunday,1,0,21:30:00,21,30,Night,7,Wilshire,0,Vehicle Theft,1900 S LONGWOOD AV,34.0375,-118.3506
1,2020-02-08,8,2,2020,Saturday,1,0,18:00:00,18,0,Evening,1,Central,1,Burglary,1000 S FLOWER ST,34.0444,-118.2628
2,2020-11-04,4,11,2020,Wednesday,0,0,17:00:00,17,0,Evening,3,Southwest,0,Vehicle Theft,1400 W 37TH ST,34.0210,-118.3002
3,2020-03-10,10,3,2020,Tuesday,0,0,20:37:00,20,37,Evening,9,Van Nuys,2,Theft,14000 RIVERSIDE DR,34.1576,-118.4387
4,2020-09-09,9,9,2020,Wednesday,0,0,06:30:00,6,30,Early Morning,4,Hollenbeck,0,Vehicle Theft,200 E AVENUE 28,34.0820,-118.2130


In [12]:
df['DATETIME'] = pd.to_datetime(df['DATE OCC'].astype(str) + ' ' + df['TIME OCC'].astype(str))

In [5]:
df['Day of Week'] = df['Day of Week'].map({
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
})

In [7]:
df['Time of Day'].unique()

array(['Night', 'Evening', 'Early Morning', 'Afternoon', 'Late Night',
       'Morning'], dtype=object)

In [8]:
df['Time of Day'] = df['Time of Day'].map({
    'Early Morning': 0,
    'Morning': 1,
    'Afternoon': 2,
    'Evening': 3,
    'Night': 4,
    'Late Night': 5
})

In [13]:
df.head()

,DATE OCC,Day,Month,Year,Day of Week,Is_Weekend,Is_Holiday,TIME OCC,Hour,Minute,Time of Day,AREA,AREA NAME,Crm Cd,Crm Cd Desc,LOCATION,LAT,LON,DATETIME
0,2020-03-01,1,3,2020,6,1,0,21:30:00,21,30,4,7,Wilshire,0,Vehicle Theft,1900 S LONGWOOD AV,34.0375,-118.3506,2020-03-01 21:30:00
1,2020-02-08,8,2,2020,5,1,0,18:00:00,18,0,3,1,Central,1,Burglary,1000 S FLOWER ST,34.0444,-118.2628,2020-02-08 18:00:00
2,2020-11-04,4,11,2020,2,0,0,17:00:00,17,0,3,3,Southwest,0,Vehicle Theft,1400 W 37TH ST,34.0210,-118.3002,2020-11-04 17:00:00
3,2020-03-10,10,3,2020,1,0,0,20:37:00,20,37,3,9,Van Nuys,2,Theft,14000 RIVERSIDE DR,34.1576,-118.4387,2020-03-10 20:37:00
4,2020-09-09,9,9,2020,2,0,0,06:30:00,6,30,0,4,Hollenbeck,0,Vehicle Theft,200 E AVENUE 28,34.0820,-118.2130,2020-09-09 06:30:00


In [14]:
features = ['DATETIME', 'Day', 'Month', 'Year', 'Day of Week', 'Hour', 'Minute', 'Is_Weekend', 'Is_Holiday', 'Time of Day', 'AREA', 'Crm Cd', 'LAT', 'LON']

In [15]:
df = df[features]

In [16]:
df.head()

,DATETIME,Day,Month,Year,Day of Week,Hour,Minute,Is_Weekend,Is_Holiday,Time of Day,AREA,Crm Cd,LAT,LON
0,2020-03-01 21:30:00,1,3,2020,6,21,30,1,0,4,7,0,34.0375,-118.3506
1,2020-02-08 18:00:00,8,2,2020,5,18,0,1,0,3,1,1,34.0444,-118.2628
2,2020-11-04 17:00:00,4,11,2020,2,17,0,0,0,3,3,0,34.0210,-118.3002
3,2020-03-10 20:37:00,10,3,2020,1,20,37,0,0,3,9,2,34.1576,-118.4387
4,2020-09-09 06:30:00,9,9,2020,2,6,30,0,0,0,4,0,34.0820,-118.2130


In [17]:
scaler_features = MinMaxScaler()
X = scaler_features.fit_transform(df[['Minute', 'Hour', 'Day', 'Month', 'Year', 'Day of Week', 'Time of Day', 'Crm Cd', 'AREA']])

In [18]:
X = pd.DataFrame(X, columns=['Minute', 'Hour', 'Day', 'Month', 'Year', 'Day of Week', 'Time of Day', 'Crm Cd', 'AREA'])
X.head()

,Minute,Hour,Day,Month,Year,Day of Week,Time of Day,Crm Cd,AREA
0,0.508475,0.913043,0.000000,0.181818,0.0,1.000000,0.8,0.0000,0.30
1,0.000000,0.782609,0.233333,0.090909,0.0,0.833333,0.6,0.0625,0.00
2,0.000000,0.739130,0.100000,0.909091,0.0,0.333333,0.6,0.0000,0.10
3,0.627119,0.869565,0.300000,0.181818,0.0,0.166667,0.6,0.1250,0.40
4,0.508475,0.260870,0.266667,0.727273,0.0,0.333333,0.0,0.0000,0.15


In [19]:
scaler_coords = MinMaxScaler()
Y = scaler_coords.fit_transform(df[['LAT', 'LON']])

In [20]:
Y = pd.DataFrame(Y, columns=['LAT', 'LON'])
Y.head()

,LAT,LON
0,0.991356,0.002671
1,0.991557,0.003411
2,0.990875,0.003096
3,0.994854,0.001929
4,0.992652,0.003831


In [21]:
data = np.hstack([X, Y])

In [22]:
def create_sequences(data, seq_len=10):
    X_seq, Y_seq = [], []
    for i in range(len(data) - seq_len):
        X_seq.append(data[i:i+seq_len, :-2])  # features
        Y_seq.append(data[i+seq_len, -2:])    # next location (LAT, LON)
    return np.array(X_seq), np.array(Y_seq)

X_seq, Y_seq = create_sequences(data, seq_len=10)

In [23]:
unique_areas = df['AREA'].unique()
G_nx = nx.complete_graph(len(unique_areas))

adj = nx.to_numpy_array(G_nx)

In [24]:
N_areas = len(unique_areas)
area_input = Input(shape=(10, X_seq.shape[2]))
x = LSTM(64, return_sequences=True)(area_input)
x = GlobalAveragePooling1D()(x)

In [25]:
gnn_input = Input(shape=(X_seq.shape[2],))
gnn_output = Dense(64, activation='relu')(gnn_input)

In [26]:
merged = Concatenate()([x, gnn_output])
out = Dense(64, activation='relu')(merged)
out = Dense(2, activation='linear')(out)

In [45]:
model = Model(inputs=[area_input, gnn_input], outputs=out)
model.compile(optimizer=Adam(0.001), loss='mse')

In [46]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 10, 64)    │     18,944 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ lstm[0][0]        │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        640 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 2)         │        130 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 27,970 (109.26 KB)

 Trainable params: 27,970 (109.26 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
area_features = np.eye(N_areas)

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X_seq, Y_seq, test_size=0.3, random_state=42)

In [63]:
gnn_inputs = X_train[:, -1] # Extract the 'AREA' column which is the last column

In [64]:
model.fit([X_train, gnn_inputs], y_train, validation_split=0.1, epochs=20, batch_size=32)

Epoch 1/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 113s 6ms/step - loss: 0.0023 - val_loss: 0.0022
Epoch 2/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 101s 5ms/step - loss: 0.0023 - val_loss: 0.0022
Epoch 3/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 104s 5ms/step - loss: 0.0023 - val_loss: 0.0022
Epoch 4/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 112s 6ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 5/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 115s 6ms/step - loss: 0.0021 - val_loss: 0.0022
Epoch 6/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 107s 5ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 7/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 113s 6ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 8/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 112s 6ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 9/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 106s 5ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 10/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 114s 6ms/step - loss: 0.0023 - val_loss: 0.0022
Epoch 11/20
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 113s 6ms/step - loss: 0.0023 - val

In [65]:
gnn_test_inputs = X_test[:, -1] # Extract the 'AREA' column which is the last column
pred_coords = model.predict([X_test, gnn_test_inputs])

9421/9421 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step


In [66]:
pred_coords_real = scaler_coords.inverse_transform(pred_coords)
true_coords_real = scaler_coords.inverse_transform(y_test)

In [67]:
m = folium.Map(location=[34.05, -118.25], zoom_start=10)

In [68]:
for i in range(20):  # plot first 20 predictions
    pred_lat, pred_lon = pred_coords_real[i]
    true_lat, true_lon = true_coords_real[i]

    folium.Marker(location=[true_lat, true_lon], popup='True', icon=folium.Icon(color='green')).add_to(m)
    folium.Marker(location=[pred_lat, pred_lon], popup='Predicted', icon=folium.Icon(color='red')).add_to(m)
    folium.PolyLine(locations=[[true_lat, true_lon], [pred_lat, pred_lon]], color='blue').add_to(m)

In [69]:
m

In [70]:
from sklearn.metrics import mean_squared_error

In [71]:
pred_coords_real = scaler_coords.inverse_transform(pred_coords)
true_coords_real = scaler_coords.inverse_transform(y_test)

In [72]:
rmse_lat = np.sqrt(mean_squared_error(true_coords_real[:, 0], pred_coords_real[:, 0]))
rmse_lon = np.sqrt(mean_squared_error(true_coords_real[:, 1], pred_coords_real[:, 1]))

In [73]:
rmse_total = np.sqrt(np.mean((true_coords_real - pred_coords_real)**2))

In [74]:
print(f"RMSE (Latitude): {rmse_lat:.5f}")
print(f"RMSE (Longitude): {rmse_lon:.5f}")
print(f"Combined RMSE: {rmse_total:.5f}")

RMSE (Latitude): 1.61016
RMSE (Longitude): 5.57956
Combined RMSE: 4.10635
